# Test Logs Generator

Author: Luca Cotti (<luca.cotti@unibs.it>)

This notebook randomly extracts a set of logs from the AIT-LDS dataset.

## Setup

In [ ]:
import csv
import os
import random
import shutil
from datetime import UTC, datetime
from pathlib import Path

import pandas as pd
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_ollama.embeddings import OllamaEmbeddings

DATASET_DIR = "out/russellmitchell_lc"
TRAIN_OUT_FILE = "out/train_logs.csv"
TEST_OUT_FILE = "out/test_logs.csv"
TRAIN_N_EVENTS = 10
TEST_N_EVENTS = 60
MAX_EVENT_CHARACTERS = 150
MAX_EVENTS_PER_FILE = 20
VECTOR_STORE_DIR = "test_logs_db"


# Reset stuff
if Path(TRAIN_OUT_FILE).exists():
    Path.unlink(TRAIN_OUT_FILE)
if Path(TEST_OUT_FILE).exists():
    Path.unlink(TEST_OUT_FILE)
if Path(VECTOR_STORE_DIR).exists():
    shutil.rmtree(VECTOR_STORE_DIR)


vector_store = Chroma(
    embedding_function=OllamaEmbeddings(model="nomic-embed-text"),
    persist_directory=VECTOR_STORE_DIR,
    collection_metadata={"hnsw:space": "cosine"},
)

## Extract logs

In [3]:
csv_files = []

# Find all CSV files in the dataset directory
for dirpath, _, filenames in os.walk(DATASET_DIR):
    # Skip monitoring directories, which do not contain logs from devices
    if "monitoring" in dirpath:
        continue

    csv_files.extend(Path(dirpath) / file for file in filenames if file.endswith(".csv"))

In [4]:
all_lines = []

for file in csv_files:
    lines = pd.read_csv(file)

    # Some CSVs do not have a "text" column for some reason, skip them
    if "text" not in lines.columns:
        continue

    lines = lines["text"].dropna().reset_index(drop=True)

    # First round of shuffling,
    # to avoid always selecting the same lines in each file
    random.shuffle(lines)

    count = 0
    for line in lines:
        # Filter out unwanted lines
        if (
            isinstance(line, str)
            and len(line) <= MAX_EVENT_CHARACTERS
            and not any(char * 6 in line for char in "-.")
            and "| Total" not in line
            and "| Value" not in line
            and "_.-" not in line
            and "-._" not in line
        ):
            all_lines.append((file, line))
            count += 1

        if count >= MAX_EVENTS_PER_FILE:
            break

# Second round of shuffling,
# to avoid always selecting lines from the same file
random.shuffle(all_lines)

selected_lines = []

for file_path, line in all_lines:
    # If a similar line is already in the vector store,
    # skip this line
    similar_results = vector_store.similarity_search_with_relevance_scores(
        "search_query: " + line,  # nomic-embed-textV1.5 requires this prefix
        k=1,
        score_threshold=0.7,
    )

    if len(similar_results) == 0:
        vector_store.add_documents([Document(page_content="search_document: " + line)])  # Again, nomic prefix
        selected_lines.append((file_path, line))

    if len(selected_lines) >= TRAIN_N_EVENTS + TEST_N_EVENTS:
        break

No relevant docs were retrieved using the relevance score threshold 0.7
No relevant docs were retrieved using the relevance score threshold 0.7
No relevant docs were retrieved using the relevance score threshold 0.7
No relevant docs were retrieved using the relevance score threshold 0.7
No relevant docs were retrieved using the relevance score threshold 0.7
No relevant docs were retrieved using the relevance score threshold 0.7
No relevant docs were retrieved using the relevance score threshold 0.7
No relevant docs were retrieved using the relevance score threshold 0.7
No relevant docs were retrieved using the relevance score threshold 0.7
No relevant docs were retrieved using the relevance score threshold 0.7


## Output

In [5]:
with Path(TRAIN_OUT_FILE).open("w", newline="", encoding="utf-8") as f:
    # Header comment
    f.write(f"# Generated on {datetime.now(tz=UTC).isoformat()}\n")

    writer = csv.writer(f)
    writer.writerow(["Device", "Application", "File", "Log Event"])

    for file_path, line in selected_lines[:TRAIN_N_EVENTS]:
        file_parts = str(file_path).split("/")

        device_name = file_parts[1]
        app_name = file_parts[2] if len(file_parts) > 3 else ""  # noqa: PLR2004
        file_name = file_parts[-1]

        writer.writerow([device_name, app_name, file_name, line])

with Path(TEST_OUT_FILE).open("w", newline="", encoding="utf-8") as f:
    # Header comment
    f.write(f"# Generated on {datetime.now(tz=UTC).isoformat()}\n")

    writer = csv.writer(f)
    writer.writerow(["Device", "Application", "File", "Log Event"])

    for file_path, line in selected_lines[TRAIN_N_EVENTS : TRAIN_N_EVENTS + TEST_N_EVENTS]:
        file_parts = str(file_path).split("/")

        device_name = file_parts[1]
        app_name = file_parts[2] if len(file_parts) > 3 else ""  # noqa: PLR2004
        file_name = file_parts[-1]

        writer.writerow([device_name, app_name, file_name, line])

# Clean up the vector store
shutil.rmtree(VECTOR_STORE_DIR)